# Tech Challenge — PNAD-COVID19 (IBGE)
### Análise para planejamento hospitalar, com a base organizada no BigQuery

Roda **tudo no Colab, de ponta a ponta**: ingestão (IBGE) → ETL em **Spark** → carga no
**BigQuery** → análises por **SQL na nuvem** → gráficos. Todas as estimativas usam o
**peso amostral (V1032)**.

**Pré-requisito:** conta Google + um projeto no Google Cloud (o *BigQuery Sandbox*
funciona sem cartão). Você informa o ID do projeto no Passo 4.

## 1 — Clonar o repositório e instalar dependências

In [ ]:
!rm -rf tech_challenge_fase_3
!git clone https://github.com/henrique819/tech_challenge_fase_3.git
%cd tech_challenge_fase_3
!pip install -q -r requirements.txt

## 2 — Ingestão: baixar os 3 meses do IBGE (set/out/nov 2020)

In [ ]:
!python download_pnad.py

## 3 — ETL em Spark: tratar e recodificar (Bronze → Silver)
Gera `dados_tratados/pnad_covid.csv`, pronto para a nuvem.

In [ ]:
!python etl_spark.py

## 4 — Carregar no BigQuery (organização em nuvem)
Edite `PROJECT` com o ID do seu projeto GCP e autorize a conta Google quando pedir.

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import sys; sys.path.insert(0, '.')
from carga_bigquery import carregar

PROJECT = "SEU_PROJETO_ID"     # <-- troque pelo ID do seu projeto GCP
DATASET, TABELA = "pnad_covid", "fato_pnad"

client = bigquery.Client(project=PROJECT)
tabela_id = carregar(client, "dados_tratados/pnad_covid.csv", PROJECT, DATASET, TABELA)

## 5 — Característica da população (SQL no BigQuery)
Responde A002 (faixa etária), A003 (sexo), A004 (raça), A005 (escolaridade).

In [ ]:
def consultar(sql):
    return client.query(sql).to_dataframe()

consultar(f'''
SELECT escolaridade, ROUND(100*SUM(peso)/SUM(SUM(peso)) OVER (),1) AS pct_pop
FROM `{tabela_id}` WHERE escolaridade IS NOT NULL
GROUP BY escolaridade ORDER BY pct_pop DESC
''')

## 6 — Sintomas clínicos (SQL no BigQuery)

In [ ]:
prev = consultar(f'''
SELECT 'Febre' s, ROUND(100*SAFE_DIVIDE(SUM(IF(sint_febre='Sim',peso,0)),SUM(IF(sint_febre IN('Sim','Não'),peso,0))),2) p FROM `{tabela_id}`
UNION ALL SELECT 'Tosse', ROUND(100*SAFE_DIVIDE(SUM(IF(sint_tosse='Sim',peso,0)),SUM(IF(sint_tosse IN('Sim','Não'),peso,0))),2) FROM `{tabela_id}`
UNION ALL SELECT 'Dor de garganta', ROUND(100*SAFE_DIVIDE(SUM(IF(sint_garganta='Sim',peso,0)),SUM(IF(sint_garganta IN('Sim','Não'),peso,0))),2) FROM `{tabela_id}`
UNION ALL SELECT 'Dif. respirar', ROUND(100*SAFE_DIVIDE(SUM(IF(sint_dificuldade_respirar='Sim',peso,0)),SUM(IF(sint_dificuldade_respirar IN('Sim','Não'),peso,0))),2) FROM `{tabela_id}`
UNION ALL SELECT 'Dor de cabeça', ROUND(100*SAFE_DIVIDE(SUM(IF(sint_dor_cabeca='Sim',peso,0)),SUM(IF(sint_dor_cabeca IN('Sim','Não'),peso,0))),2) FROM `{tabela_id}`
UNION ALL SELECT 'Perda olfato/paladar', ROUND(100*SAFE_DIVIDE(SUM(IF(sint_perda_olfato_paladar='Sim',peso,0)),SUM(IF(sint_perda_olfato_paladar IN('Sim','Não'),peso,0))),2) FROM `{tabela_id}`
ORDER BY p
''')
prev

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(prev['s'], prev['p'], color='#c0392b')
ax.set_xlabel('Prevalência ponderada (%)'); ax.set_title('Sintomas (set–nov/2020) — via BigQuery')
plt.tight_layout(); plt.show()

In [ ]:
# Internação por faixa etária (B005 x A002) — risco por idade
consultar(f'''
SELECT faixa_etaria,
  ROUND(100*SAFE_DIVIDE(SUM(IF(internado='Sim',peso,0)),SUM(IF(internado IN('Sim','Não'),peso,0))),2) AS taxa_internacao
FROM `{tabela_id}` WHERE faixa_etaria IS NOT NULL
GROUP BY faixa_etaria ORDER BY faixa_etaria
''')

## 7 — Comportamento da população (SQL no BigQuery)

In [ ]:
# Isolamento ao longo dos 3 meses (B011)
consultar(f'''
SELECT mes_nome, grau_isolamento, ROUND(SUM(peso)) AS pop
FROM `{tabela_id}` WHERE grau_isolamento IS NOT NULL
GROUP BY mes_nome, grau_isolamento ORDER BY mes_nome, grau_isolamento
''')

In [ ]:
# Testagem por escolaridade (B008 x A005)
consultar(f'''
SELECT escolaridade,
  ROUND(100*SAFE_DIVIDE(SUM(IF(fez_teste='Sim',peso,0)),SUM(IF(fez_teste IN('Sim','Não'),peso,0))),1) AS taxa_teste
FROM `{tabela_id}` WHERE escolaridade IS NOT NULL
GROUP BY escolaridade ORDER BY taxa_teste
''')

## 8 — Características econômicas (SQL no BigQuery)
Auxílio emergencial (**D0051**) por posição na ocupação, e isolamento rigoroso por faixa de renda.

In [ ]:
# Auxílio emergencial por posição na ocupação (D0051 x C007)
# Denominador = todos da ocupação (a D0051 tem ~33% de nulos: nulo/Não = "não recebeu").
consultar(f'''
SELECT posicao_ocupacao,
  ROUND(100*SAFE_DIVIDE(SUM(IF(auxilio_emergencial='Sim',peso,0)), SUM(peso)),1) AS pct_auxilio
FROM `{tabela_id}` WHERE posicao_ocupacao IS NOT NULL
GROUP BY posicao_ocupacao ORDER BY pct_auxilio DESC
''')

In [ ]:
# Isolamento rigoroso por faixa de renda (C01012 x B011) — SÓ OCUPADOS (rendimento > 0)
# As faixas de renda só existem para quem tem rendimento; restringir evita o confundidor
# de 'Sem renda' (não-ocupados que ficam em casa por não trabalhar).
consultar(f'''
WITH base AS (
  SELECT peso, grau_isolamento,
    CASE WHEN rendimento<=1045 THEN 'Até 1 SM' WHEN rendimento<=2090 THEN '1-2 SM'
         WHEN rendimento<=5225 THEN '2-5 SM' ELSE '5+ SM' END AS faixa_renda
  FROM `{tabela_id}` WHERE rendimento IS NOT NULL AND rendimento>0)
SELECT faixa_renda,
  ROUND(100*SAFE_DIVIDE(SUM(IF(grau_isolamento='Ficou rigorosamente isolado',peso,0)),SUM(peso)),1) AS pct_isolado_rigoroso
FROM base WHERE grau_isolamento IS NOT NULL
GROUP BY faixa_renda ORDER BY pct_isolado_rigoroso
''')

## 9 — Ações ao hospital em caso de novo surto

A partir das análises acima (a leitura dos números reais é sua):

1. **Triagem por sintoma-marcador** — prioridade para dificuldade de respirar e perda de olfato/paladar.
2. **Leitos por faixa etária** — reserva proporcional ao risco, maior para idosos.
3. **Testagem dirigida** — mutirões onde a testagem é menor (baixa escolaridade/renda).
4. **Retaguarda do SUS** — dimensionar pela cobertura de plano de saúde.
5. **Gatilho de renda** — isolamento não se sustenta sem transferência de renda a informais.
6. **Vigilância contínua** — reaproveitar este pipeline para alertas por sintoma-marcador.

## 10 (opcional) — Liberar leitura do dataset
Permite que outras contas Google consultem a tabela (cada uma usa o próprio projeto para faturar a query).

In [ ]:
# from carga_bigquery import tornar_publico
# tornar_publico(client, PROJECT, DATASET)